# Seminar Notebook 3.5: Naive Bayes by Hand

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook walks through the Naive Bayes classification procedure manually, step by step.

## Set up steps

### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
if not os.path.exists(sdir):
    os.makedirs(sdir)

### Loading the data

We will use the corpus of tweets posted by the major candidates for the 2016 U.S. presidential election, for which we created a DFM in `01-making-dfm.ipynb`. 

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse

# Load the sparse DFM
dfm = sparse.load_npz(os.path.join(sdir, "candidate-dfm.npz"))
print(f"DFM shape: {dfm.shape}")

# Load the vocabulary
with open(os.path.join(sdir, "candidate-dfm-features.txt"), "r") as f:
    vocabulary = f.read().split("\n")
print(f"Number of features: {len(vocabulary)}")

# Load the corpus metadata
corpus = pd.read_csv(os.path.join(sdir, "candidate-corpus.csv"))
corpus.head()

As we did in the previous notebook, we need to create the document-level labels and split our data into training and test sets.

In [ ]:
# Create Clinton label based on screen_name
corpus["Clinton"] = corpus["screen_name"].apply(lambda x: 1 if "Clinton" in x else 0)
print(corpus["Clinton"].value_counts())

We will use 75% for training and 25% for test.

In [ ]:
from sklearn.model_selection import train_test_split

dfm_train, dfm_test, labels_train, labels_test = train_test_split(
    dfm, corpus["Clinton"],
    test_size=0.25,
    random_state=755
)

In [ ]:
print(f"Training labels: {pd.Series(labels_train).value_counts().to_dict()}")
print(f"Test labels: {pd.Series(labels_test).value_counts().to_dict()}")

## Naive Bayes "by hand"

To solidify your understanding of how Naive Bayes works, let's walk through the classification procedure manually. This is useful for understanding the mechanics and for exam preparation.

Recall the formula from lecture:
$$
\Pr(\pi_{ik} = 1 | D_i) \propto \hat{\alpha}_k \prod_{j=1}^J \left(\hat{\mu}_{kj}^{W_{ij}}\right)
$$

Where:
- $\hat{\alpha}_k$ is the proportion of documents in class $k$
- $\hat{\mu}_{kj}$ is the probability of word $j$ given class $k$
- $W_{ij}$ is the count of word $j$ in document $i$

### Step 1: Estimate class proportions ($\hat{\alpha}_k$)

First, we calculate the proportion of documents in each class.

In [ ]:
# Calculate class proportions from training data
n_train = len(labels_train)
alpha_C = np.sum(labels_train == 1) / n_train
alpha_NC = np.sum(labels_train == 0) / n_train

print(f"alpha_C (proportion Clinton): {alpha_C:.3f}")
print(f"alpha_NC (proportion Not Clinton): {alpha_NC:.3f}")

### Step 2: Collapse DFM by class

For Naive Bayes, we treat all documents in a class as if they were generated from the same language model. So we sum word counts across all documents in each class.

In [ ]:
# Sum word counts by class
dfm_C = dfm_train[(labels_train == 1).values, :].sum(axis=0).A1  # .A1 converts to 1D array
dfm_NC = dfm_train[(labels_train == 0).values, :].sum(axis=0).A1

print(f"Total tokens in Clinton documents: {dfm_C.sum():.0f}")
print(f"Total tokens in Not Clinton documents: {dfm_NC.sum():.0f}")

### Step 3: Estimate word probabilities ($\hat{\mu}_{kj}$)

For each word $j$ and class $k$, we calculate the probability of that word given the class. We add a Laplace smoother to handle words that don't appear in a class.

$$
\hat{\mu}_{kj} = \frac{c + \sum_i \pi_{ik} W_{ij}}{\sum_j \left(c + \sum_i \pi_{ik} W_{ij}\right)}
$$

With Laplace smoother $c=1$, this simplifies to:
$$
\hat{\mu}_{kj} = \frac{1 + \text{count of word } j \text{ in class } k}{J + \text{total tokens in class } k}
$$

In [ ]:
# Laplace smoother
c = 1
J = len(vocabulary)

# Calculate mu for each class
mu_C = (c + dfm_C) / (J * c + dfm_C.sum())
mu_NC = (c + dfm_NC) / (J * c + dfm_NC.sum())

print(f"Number of features (J): {J}")
print(f"Sum of mu_C (should be ~1): {mu_C.sum():.6f}")
print(f"Sum of mu_NC (should be ~1): {mu_NC.sum():.6f}")

Let's look at the estimated probabilities for a few example words.

In [ ]:
# Look at some example words
example_words = ["hillari", "trump", "vote", "women"]

for word in example_words:
    if word in vocabulary:
        idx = vocabulary.index(word)
        print(f"{word}: mu_C = {mu_C[idx]:.6f}, mu_NC = {mu_NC[idx]:.6f}")

Now we can classify a document by calculating the probability of each class and choosing the one with the higher probability. Let's classify the first document in the test set.

In [ ]:
# Get the first test document
doc_idx = 0
doc = dfm_test[doc_idx, :].toarray().flatten()
true_label = labels_test.iloc[doc_idx]

print(f"True label = {true_label}")
print(f"Total tokens in document: {doc.sum():.0f}")

We calculate $\hat{\alpha}_k \prod_{j} \hat{\mu}_{kj}^{W_{ij}}$ for each class. 

**Computational note:** In lecture, we worked with small examples where this product is easy to compute. Here, we have nearly 2,400 features, so the $\hat{\mu}_{kj}$ values are tiny (around 0.0001 or smaller). Multiplying thousands of such small numbers together causes numerical underflow---the computer rounds to zero. To avoid this, we convert to log space:

$$
\log \Pr(k | D_i) \propto \log \hat{\alpha}_k + \sum_{j} W_{ij} \log \hat{\mu}_{kj}
$$

This is just a computational trick; the classification decision is the same.

In [ ]:
# Calculate log-probabilities for each class
log_prob_C = np.log(alpha_C) + np.sum(doc * np.log(mu_C))
log_prob_NC = np.log(alpha_NC) + np.sum(doc * np.log(mu_NC))

print(f"Log probability of Clinton: {log_prob_C:.2f}")
print(f"Log probability of Not Clinton: {log_prob_NC:.2f}")

# Classify based on higher probability
predicted = 1 if log_prob_C > log_prob_NC else 0
print(f"\nPredicted: {predicted}, True: {true_label}")

### Step 5: Classify all test documents

Let's apply this to all test documents and compare with sklearn's implementation.

In [ ]:
# Classify all test documents
manual_preds = []

for i in range(dfm_test.shape[0]):
    doc = dfm_test[i, :].toarray().flatten()
    
    log_prob_C = np.log(alpha_C) + np.sum(doc * np.log(mu_C))
    log_prob_NC = np.log(alpha_NC) + np.sum(doc * np.log(mu_NC))
    
    predicted = 1 if log_prob_C > log_prob_NC else 0
    manual_preds.append(predicted)

manual_preds = np.array(manual_preds)

In [ ]:
# Compare with sklearn predictions
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB(alpha=1.0)
nb.fit(dfm_train, labels_train)
sklearn_preds = nb.predict(dfm_test)

print("Manual vs sklearn predictions:")
print(f"Agreement: {np.mean(manual_preds == sklearn_preds):.1%}")

# Create confusion matrix for manual predictions
manual_cm = pd.crosstab(labels_test, manual_preds,
                        rownames=["True"], colnames=["Predicted"])
print("\nManual Naive Bayes confusion matrix:")
print(manual_cm)